# SPANet behavior on VBF, ggF and tt_sl

This notebook compares the stored ColumnFlow SPANet outputs across three reduced-event samples:

- `spanet_v9`: VBF HH signal
- `ggf/spanet_v9`: ggF HH signal
- `ttsl/spanet_v9`: semileptonic ttbar background

The purpose here is not truth matching. The purpose is to check whether the stored SPANet selections and scores behave sensibly across different physics processes.

In [1]:
from pathlib import Path
import glob

import awkward as ak
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["axes.titlesize"] = 14
plt.rcParams["axes.labelsize"] = 12

BASE_DIR = Path("/home/norman/UHHAnalysis/test_tt")

SAMPLES = {
    "VBF HH": BASE_DIR / "spanet_v9",
    "ggF HH": BASE_DIR / "ggf" / "spanet_v9",
    "tt_sl": BASE_DIR / "ttsl" / "spanet_v9",
}

def parquet_files(sample_dir):
    files = sorted(glob.glob(str(sample_dir / "events_*.parquet")))
    if not files:
        raise FileNotFoundError(f"No parquet files found in {sample_dir}")
    return files

def load_events(sample_dir):
    return ak.from_parquet(parquet_files(sample_dir))

def has_two(collection):
    return ak.to_numpy(ak.num(collection, axis=1) >= 2)

def first_of_pair(collection, field):
    values = ak.pad_none(collection[field], 1, clip=True)
    return ak.to_numpy(ak.fill_none(values[:, 0], np.nan))

def pair_mean(collection, field):
    return ak.to_numpy(ak.fill_none(ak.mean(collection[field], axis=1), np.nan))

def pair_max(collection, field):
    return ak.to_numpy(ak.fill_none(ak.max(collection[field], axis=1), np.nan))

def valid_scores(x):
    return x[np.isfinite(x)]

def frac_above(values, thr):
    values = values[np.isfinite(values)]
    if len(values) == 0:
        return np.nan
    return 100.0 * np.mean(values > thr)


In [2]:
events_by_sample = {name: load_events(path) for name, path in SAMPLES.items()}

pd.DataFrame([
    {
        "Sample": name,
        "Directory": str(path),
        "Parquet files": len(parquet_files(path)),
        "Events": len(events_by_sample[name]),
    }
    for name, path in SAMPLES.items()
])

FileNotFoundError: No parquet files found in /home/norman/UHHAnalysis/test_tt/spanet_v9

## Event-level selection summary

This table answers the simplest behavior questions:

- how often the baseline taggers produce an HH->bb pair or a VBF pair,
- how often SPANet produces an HH->bb pair or a VBF pair,
- what the stored SPANet detection and assignment scores look like in each sample.

In [ ]:
rows = []

for sample_name, events in events_by_sample.items():
    baseline_hhb = has_two(events.HHBJet)
    baseline_vbf = has_two(events.VBFJet)
    spanet_hhb = has_two(events.SPANetHHBJet)
    spanet_vbf = has_two(events.SPANetVBFJet)

    hhb_det = first_of_pair(events.SPANetHHBJet, "spanet_higgs_bb_detection_score")
    vbf_det = first_of_pair(events.SPANetVBFJet, "spanet_vbf_detection_score")
    hhb_assign = first_of_pair(events.SPANetHHBJet, "spanet_hhb_assignment_score")
    vbf_assign = first_of_pair(events.SPANetVBFJet, "spanet_vbf_assignment_score")

    rows.append({
        "Sample": sample_name,
        "Events": len(events),
        "Baseline HH->bb pair [%]": 100.0 * baseline_hhb.mean(),
        "SPANet HH->bb pair [%]": 100.0 * spanet_hhb.mean(),
        "Baseline VBF pair [%]": 100.0 * baseline_vbf.mean(),
        "SPANet VBF pair [%]": 100.0 * spanet_vbf.mean(),
        "SPANet HH det mean": np.nanmean(hhb_det),
        "SPANet VBF det mean": np.nanmean(vbf_det),
        "SPANet HH assign mean": np.nanmean(hhb_assign),
        "SPANet VBF assign mean": np.nanmean(vbf_assign),
    })

selection_summary = pd.DataFrame(rows)
selection_summary

## Detection-score threshold summary

For signal-vs-background behavior, these fractions are often more informative than just the mean score.

Interpretation:
- HH detection should stay high in both VBF HH and ggF HH.
- VBF detection should be clearly higher in VBF HH than in ggF HH and tt_sl.
- Both detection scores should be lower in tt_sl than in signal.

In [ ]:
thresholds = [0.1, 0.5, 0.9]
rows = []

for sample_name, events in events_by_sample.items():
    hhb_det = first_of_pair(events.SPANetHHBJet, "spanet_higgs_bb_detection_score")
    vbf_det = first_of_pair(events.SPANetVBFJet, "spanet_vbf_detection_score")
    
    row = {"Sample": sample_name}
    for thr in thresholds:
        row[f"HH det > {thr}"] = frac_above(hhb_det, thr)
        row[f"VBF det > {thr}"] = frac_above(vbf_det, thr)
    rows.append(row)

threshold_summary = pd.DataFrame(rows)
threshold_summary

## Baseline selected-pair scores

These are the tagger scores on the already selected baseline jet pairs.
They are useful as context when comparing baseline behavior to SPANet behavior across the same samples.

In [ ]:
rows = []

for sample_name, events in events_by_sample.items():
    rows.append({
        "Sample": sample_name,
        "Baseline HH hhbtag mean": np.nanmean(pair_mean(events.HHBJet, "hhbtag")),
        "Baseline VBF vbfjtag mean": np.nanmean(pair_mean(events.VBFJet, "vbfjtag")),
        "SPANet HH selected hhbtag mean": np.nanmean(pair_mean(events.SPANetHHBJet, "hhbtag")),
        "SPANet VBF selected vbfjtag mean": np.nanmean(pair_mean(events.SPANetVBFJet, "vbfjtag")),
    })

baseline_score_context = pd.DataFrame(rows)
baseline_score_context

## Main behavior plots

In [ ]:
plot_rows = []
for sample_name, events in events_by_sample.items():
    plot_rows.extend([
        {"Sample": sample_name, "Algorithm": "Baseline", "Target": "HH->bb", "Fraction [%]": 100.0 * has_two(events.HHBJet).mean()},
        {"Sample": sample_name, "Algorithm": "SPANet", "Target": "HH->bb", "Fraction [%]": 100.0 * has_two(events.SPANetHHBJet).mean()},
        {"Sample": sample_name, "Algorithm": "Baseline", "Target": "VBF", "Fraction [%]": 100.0 * has_two(events.VBFJet).mean()},
        {"Sample": sample_name, "Algorithm": "SPANet", "Target": "VBF", "Fraction [%]": 100.0 * has_two(events.SPANetVBFJet).mean()},
    ])

plot_df = pd.DataFrame(plot_rows)

fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=True)
for ax, target in zip(axes, ["HH->bb", "VBF"]):
    sub = plot_df[plot_df["Target"] == target]
    sns.barplot(data=sub, x="Sample", y="Fraction [%]", hue="Algorithm", ax=ax)
    ax.set_title(f"Selected {target} pair fraction")
    ax.set_ylim(0, 105)
    ax.legend(loc="best")
plt.tight_layout()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for sample_name, events in events_by_sample.items():
    hhb_det = valid_scores(first_of_pair(events.SPANetHHBJet, "spanet_higgs_bb_detection_score"))
    vbf_det = valid_scores(first_of_pair(events.SPANetVBFJet, "spanet_vbf_detection_score"))
    hhb_assign = valid_scores(first_of_pair(events.SPANetHHBJet, "spanet_hhb_assignment_score"))
    vbf_assign = valid_scores(first_of_pair(events.SPANetVBFJet, "spanet_vbf_assignment_score"))

    axes[0, 0].hist(hhb_det, bins=40, histtype="step", density=True, linewidth=2, label=sample_name)
    axes[0, 1].hist(vbf_det, bins=40, histtype="step", density=True, linewidth=2, label=sample_name)
    axes[1, 0].hist(hhb_assign, bins=40, histtype="step", density=True, linewidth=2, label=sample_name)
    axes[1, 1].hist(vbf_assign, bins=40, histtype="step", density=True, linewidth=2, label=sample_name)

axes[0, 0].set_title("SPANet HH detection score")
axes[0, 1].set_title("SPANet VBF detection score")
axes[1, 0].set_title("SPANet HH assignment score")
axes[1, 1].set_title("SPANet VBF assignment score")

for ax in axes.flat:
    ax.set_xlabel("Score")
    ax.set_ylabel("Density")
    ax.legend(loc="best")

plt.tight_layout()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=False)

for sample_name, events in events_by_sample.items():
    base_hhb = valid_scores(pair_mean(events.HHBJet, "hhbtag"))
    spanet_hhb = valid_scores(pair_mean(events.SPANetHHBJet, "hhbtag"))
    base_vbf = valid_scores(pair_mean(events.VBFJet, "vbfjtag"))
    spanet_vbf = valid_scores(pair_mean(events.SPANetVBFJet, "vbfjtag"))

    axes[0].hist(base_hhb, bins=40, histtype="step", density=True, linewidth=2, label=f"{sample_name} baseline")
    axes[0].hist(spanet_hhb, bins=40, histtype="step", density=True, linewidth=2, linestyle="--", label=f"{sample_name} SPANet")
    axes[1].hist(base_vbf, bins=40, histtype="step", density=True, linewidth=2, label=f"{sample_name} baseline")
    axes[1].hist(spanet_vbf, bins=40, histtype="step", density=True, linewidth=2, linestyle="--", label=f"{sample_name} SPANet")

axes[0].set_title("Selected HH->bb pair hhbtag")
axes[1].set_title("Selected VBF pair vbfjtag")

for ax in axes:
    ax.set_xlabel("Score")
    ax.set_ylabel("Density")
    ax.legend(loc="best", fontsize=9)

plt.tight_layout()

## What to look for

Expected qualitative behavior:

- `VBF HH`: high HH detection, high VBF detection, frequent HH and VBF pair selection.
- `ggF HH`: high HH detection, much weaker VBF detection than VBF HH.
- `tt_sl`: lower HH detection than signal and low VBF detection.

If the VBF detection score stays large for `ggF` or `tt_sl`, that would suggest the model is over-assigning VBF structure where it should not.